# Bayesian Hyperparameter Space Optimization and Automated Model Auditing Pipeline

In [1]:
%load_ext autoreload
%autoreload 2

import os
import mlflow
import optuna
from optuna.integration.mlflow import MLflowCallback
from optuna.pruners import MedianPruner
import pandas as pd

from src import optuna_optimization as utils

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Path Configuration & MLflow Backend Binding

In [2]:
RANDOM_STATE = 42
N_SPLITS = 5
EXPERIMENT_NAME = "customer-churn-optuna"

ROOT_DIR = os.path.abspath("../")
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

# Set the tracking URI immediately to lock it to SQLite
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

# Explicitly create the experiment with your designated mlartifacts folder path
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    mlflow.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=f"file://{ARTIFACTS_DIR}"
    )

# Active the experiment scope
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='file:///Users/hector.vargas/repos/ml_hands_on_project/mlartifacts', creation_time=1780446392364, experiment_id='4', last_update_time=1780446392364, lifecycle_stage='active', name='customer-churn-optuna', tags={}, trace_location=None, workspace='default'>

## 2. Custom Source Code Ingestion

In [3]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

In [4]:
# Mapping Runtime Feature Schemas
nomod_columns = ['HasCrCard', 'IsActiveMember']
dummyfy_columns = ['Card Type', 'NumOfProducts', 'Geography', 'Gender']
norm_std_columns = ['Balance', 'Point Earned', 'CreditScore', 'Age', 'Tenure', 'Satisfaction Score', 'EstimatedSalary']

## 2. Orchestrate and Initialize Search Parameter Studies

In [5]:
# Initialize Integrated MLflow Tracking Integration Callbacks
mlflow_callback = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="average_precision",
    create_experiment=False, # Keeps Optuna from trying to override our custom artifact path
    mlflow_kwargs={"nested": True}
)

study = optuna.create_study(
    direction='maximize',
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=0)
)

/var/folders/bv/50x24wc545x5mclk_t88ryrc0000gn/T/ipykernel_12326/1472475305.py:2: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
[I 2026-06-02 18:27:45,588] A new study created in memory with name: no-name-5601a31c-77be-4788-b838-26af6a2e7c5b


In [6]:
# Instantiate the cross-validation objective callable
objective_function = utils.ObjectiveCV(
    X=X_train, y=y_train, 
    cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns,
    n_splits=N_SPLITS, random_state=RANDOM_STATE
)

## 3. Run Optimization Search Workspace Execution Window

In [7]:
with mlflow.start_run(run_name="optuna_search_parent"):
    
    study.optimize(
        objective_function,
        n_trials=500,
        callbacks=[mlflow_callback]
    )

    # Log global best performance summary attributes
    mlflow.log_metric("best_auc", study.best_value)
    mlflow.log_params(study.best_params)

    # Reconstruct and serialize top performing candidate artifacts pipeline
    best_pipeline = utils.build_pipeline(
        study.best_trial, 
        cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns,
        random_state=RANDOM_STATE
    )

    # This internally calls artifact logging functions; they will now map straight to mlartifacts/
    utils.evaluate_and_log_best_model(
        best_pipeline=best_pipeline,
        X_train=X_train, y_train=y_train,
        X_test=X_test, y_test=y_test,
        cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns
    )

[I 2026-06-02 18:27:47,738] Trial 0 finished with value: 0.6994941348687105 and parameters: {'scaler': 'std', 'encoder': 'drop_first', 'model': 'xgb', 'xgb_n_estimators': 911, 'xgb_max_depth': 6, 'xgb_learning_rate': 0.06193403057044833, 'xgb_subsample': 0.9935786314247885, 'xgb_colsample_bytree': 0.5587156488652159, 'xgb_min_child_weight': 5, 'xgb_gamma': 4.622707201091718, 'xgb_reg_alpha': 1.4371698139771932e-05, 'xgb_reg_lambda': 1.8191624821193573e-05, 'xgb_scale_pos_weight': 2.0889778194938926}. Best is trial 0 with value: 0.6994941348687105.
[I 2026-06-02 18:27:49,896] Trial 1 finished with value: 0.6792197054125924 and parameters: {'scaler': 'minmax', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 416, 'rf_max_depth': 9, 'rf_min_samples_split': 14, 'rf_min_samples_leaf': 7}. Best is trial 0 with value: 0.6994941348687105.
[I 2026-06-02 18:27:51,271] Trial 2 finished with value: 0.7003845084314897 and parameters: {'scaler': 'minmax', 'encoder': 'no_drop', 'model': 'xg

## 4. Display Session Optimization Diagnostics Results

In [8]:
print(f"\nTop Optimization Average Precision Score Target: {study.best_value:.4f}")
print("\nOptimal Parameter Combinations Selected:")
for parameter_key, parameter_value in study.best_params.items():
    print(f" * {parameter_key}: {parameter_value}")


Top Optimization Average Precision Score Target: 0.7075

Optimal Parameter Combinations Selected:
 * scaler: minmax
 * encoder: drop_first
 * model: xgb
 * xgb_n_estimators: 740
 * xgb_max_depth: 5
 * xgb_learning_rate: 0.03612453446877948
 * xgb_subsample: 0.9074262034930778
 * xgb_colsample_bytree: 0.6470595692288219
 * xgb_min_child_weight: 3
 * xgb_gamma: 4.260119163678167
 * xgb_reg_alpha: 0.00019922330312753135
 * xgb_reg_lambda: 9.868376530013087e-05
 * xgb_scale_pos_weight: 1.714635351221534


Best average_precision: **0.7077109199922507**

Optimal Parameter Combinations Selected:
 * scaler: robust
 * encoder: drop_first
 * model: xgb
 * xgb_n_estimators: 588
 * xgb_max_depth: 5
 * xgb_learning_rate: 0.06090819562411312
 * xgb_subsample: 0.967454071450258
 * xgb_colsample_bytree: 0.6262720546196823
 * xgb_min_child_weight: 3
 * xgb_gamma: 3.2784302986561005
 * xgb_reg_alpha: 0.0006480121274906255
 * xgb_reg_lambda: 7.441405323298544e-05
 * xgb_scale_pos_weight: 1.8090219464356987


Top Optimization Average Precision Score Target: **0.7079**

Optimal Parameter Combinations Selected:
 * scaler: minmax
 * encoder: drop_first
 * model: xgb
 * xgb_n_estimators: 967
 * xgb_max_depth: 5
 * xgb_learning_rate: 0.05903303186469791
 * xgb_subsample: 0.9204856064383659
 * xgb_colsample_bytree: 0.6118872682736142
 * xgb_min_child_weight: 2
 * xgb_gamma: 4.2451076897824676
 * xgb_reg_alpha: 0.00019575646965291755
 * xgb_reg_lambda: 0.0005189779456101976
 * xgb_scale_pos_weight: 1.6516119336541517